In [10]:
import pandas as pd
from urllib import request

data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

lines = data.read().decode("utf-8").split("\n")[2:] #skiping first two lines as they just have metadata

playlists = [s.rstrip().split() for s in lines if len(s.split())>1] #remove playlist that have less than 2 songs

#load metadata seprate

songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_lines = songs_file.read().decode("utf-8").split("\n")
songs = [s.rstrip().split('\t') for s in songs_lines if '\t' in s] # Use songs_lines and ensure valid lines

songs_df = pd.DataFrame(data=songs, columns= ['id', 'title', 'artist'])
songs_df['id'] = songs_df['id'].str.strip() # Strip whitespace from the 'id' column
songs_df = songs_df.set_index('id')

In [11]:
print(songs_df)

                                                    title             artist
id                                                                          
0                            Gucci Time (w\/ Swizz Beatz)         Gucci Mane
1       Aston Martin Music (w\/ Drake & Chrisette Mich...          Rick Ross
2                           Get Back Up (w\/ Chris Brown)               T.I.
3                      Hot Toddy (w\/ Jay-Z & Ester Dean)              Usher
4                                            Whip My Hair             Willow
...                                                   ...                ...
75257                              Dearest (I'm So Sorry)  Picture Me Broken
75258                                           USA Today       Alan Jackson
75259                                           Superstar          Raul Malo
75260                                 Romancin' The Blues      Giacomo Gates
75261                                        Inner Change    The Jazzmasters

In [4]:
print(playlists[0])

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43']


In [6]:
#training model
!pip install gensim
from gensim.models import Word2Vec

model = Word2Vec(
    playlists, vector_size=100, window=5,negative=50, min_count=1, workers=4
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 20.2 MB/s eta 0:00:00


In [18]:
song_id = 2172

model.wv.most_similar(positive=str(song_id)) # will give list of songs whose embeddings are most similar

[('10094', 0.9988934397697449),
 ('3148', 0.9988848567008972),
 ('3119', 0.9981640577316284),
 ('6660', 0.9980177879333496),
 ('2902', 0.9980047345161438),
 ('2849', 0.9978917837142944),
 ('3079', 0.997626781463623),
 ('3080', 0.9973825216293335),
 ('2014', 0.9972028732299805),
 ('3167', 0.997198224067688)]

In [19]:
print(songs_df.iloc[2172] )

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [20]:
import numpy as np
similar_songs = np.array(model.wv.most_similar(positive=str(song_id)))[:,0]
songs_df.iloc[similar_songs]

,title,artist
id,,
10094,I Don't Believe In Love,Queensryche
3148,Big City Nights,Scorpions
3119,There's Only One Way To Rock,Sammy Hagar
6660,Slow An' Easy,Whitesnake
2902,Jailbreak,Thin Lizzy
2849,Run To The Hills,Iron Maiden
3079,The Trees,Rush
3080,Man On The Silver Mountain,Rainbow
2014,Youth Gone Wild,Skid Row
